In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv(
    "../datasets/processed/dividend_puzzle_fundamental_dataset.csv"
)

print("Shape:", data.shape)
print(data.columns.tolist())

Shape: (110, 17)
['company', 'ticker', 'year', 'year_end_price', 'dividend_per_share', 'dividend_yield', 'dividend_growth', 'dividend_growth_category', 'annual_stock_return', 'return_category', 'total_return', 'return_volatility', 'average_return', 'dividend_consistency', 'revenue', 'net_income', 'total_assets']


In [2]:
fundamentals = pd.read_csv(
    "../datasets/raw/fundamentals_real.csv"
)

print("Fundamentals shape:", fundamentals.shape)
print(fundamentals.columns.tolist())

Fundamentals shape: (46, 5)
['ticker', 'year', 'revenue', 'net_income', 'total_assets']


In [3]:
data["ticker"] = (
    data["ticker"]
    .astype(str)
    .str.strip()
    .str.upper()
)

fundamentals["ticker"] = (
    fundamentals["ticker"]
    .astype(str)
    .str.strip()
    .str.upper()
)

data["year"] = pd.to_numeric(
    data["year"],
    errors="coerce"
)

fundamentals["year"] = pd.to_numeric(
    fundamentals["year"],
    errors="coerce"
)

print("Keys cleaned successfully.")

Keys cleaned successfully.


In [4]:
common = data.merge(
    fundamentals[["ticker", "year"]],
    on=["ticker", "year"],
    how="inner"
)

print("Matching company-year rows:", len(common))

display(
    common[
        ["ticker", "year"]
    ].sort_values(["ticker", "year"])
)

Matching company-year rows: 43


,ticker,year
0,AAPL,2021
1,AAPL,2022
2,AAPL,2023
3,AAPL,2024
4,AAPL,2025
5,JNJ,2021
6,JNJ,2022
7,JNJ,2023
8,JNJ,2024
9,JNJ,2025


In [5]:
data = data.merge(
    fundamentals[
        [
            "ticker",
            "year",
            "revenue",
            "net_income",
            "total_assets"
        ]
    ],
    on=["ticker", "year"],
    how="left"
)

print("Final shape after merge:", data.shape)

print(
    data.columns.tolist()
)

Final shape after merge: (110, 20)
['company', 'ticker', 'year', 'year_end_price', 'dividend_per_share', 'dividend_yield', 'dividend_growth', 'dividend_growth_category', 'annual_stock_return', 'return_category', 'total_return', 'return_volatility', 'average_return', 'dividend_consistency', 'revenue_x', 'net_income_x', 'total_assets_x', 'revenue_y', 'net_income_y', 'total_assets_y']


In [6]:
print(
    data[
        [
            "ticker",
            "year",
            "revenue",
            "net_income",
            "total_assets"
        ]
    ].head(20)
)

KeyError: "['revenue', 'net_income', 'total_assets'] not in index"

In [7]:
print("ALL COLUMNS:")
for i, col in enumerate(data.columns):
    print(i, repr(col))

ALL COLUMNS:
0 'company'
1 'ticker'
2 'year'
3 'year_end_price'
4 'dividend_per_share'
5 'dividend_yield'
6 'dividend_growth'
7 'dividend_growth_category'
8 'annual_stock_return'
9 'return_category'
10 'total_return'
11 'return_volatility'
12 'average_return'
13 'dividend_consistency'
14 'revenue_x'
15 'net_income_x'
16 'total_assets_x'
17 'revenue_y'
18 'net_income_y'
19 'total_assets_y'


In [8]:
# Use the newly merged fundamental values
data["revenue"] = data["revenue_y"]
data["net_income"] = data["net_income_y"]
data["total_assets"] = data["total_assets_y"]

print("Columns fixed successfully!")

print(
    data[
        ["ticker", "year", "revenue", "net_income", "total_assets"]
    ].head(20).to_string(index=False)
)

Columns fixed successfully!
ticker  year      revenue   net_income  total_assets
  AAPL  2015          NaN          NaN           NaN
  AAPL  2016          NaN          NaN           NaN
  AAPL  2017          NaN          NaN           NaN
  AAPL  2018          NaN          NaN           NaN
  AAPL  2019          NaN          NaN           NaN
  AAPL  2020          NaN          NaN           NaN
  AAPL  2021          NaN          NaN           NaN
  AAPL  2022 3.943280e+11 9.980300e+10  3.527550e+11
  AAPL  2023 3.832850e+11 9.699500e+10  3.525830e+11
  AAPL  2024 3.910350e+11 9.373600e+10  3.649800e+11
  AAPL  2025 4.161610e+11 1.120100e+11  3.592410e+11
   JNJ  2015          NaN          NaN           NaN
   JNJ  2016          NaN          NaN           NaN
   JNJ  2017          NaN          NaN           NaN
   JNJ  2018          NaN          NaN           NaN
   JNJ  2019          NaN          NaN           NaN
   JNJ  2020          NaN          NaN           NaN
   JNJ  2021      

In [9]:
data = data.drop(
    columns=[
        "revenue_x",
        "net_income_x",
        "total_assets_x",
        "revenue_y",
        "net_income_y",
        "total_assets_y"
    ]
)

print(data.columns.tolist())

['company', 'ticker', 'year', 'year_end_price', 'dividend_per_share', 'dividend_yield', 'dividend_growth', 'dividend_growth_category', 'annual_stock_return', 'return_category', 'total_return', 'return_volatility', 'average_return', 'dividend_consistency', 'revenue', 'net_income', 'total_assets']


In [10]:
data["profit_margin"] = np.where(
    data["revenue"] != 0,
    (data["net_income"] / data["revenue"]) * 100,
    np.nan
)

print("Profit Margin calculated.")

Profit Margin calculated.


In [11]:
data["roa"] = np.where(
    data["total_assets"] != 0,
    (data["net_income"] / data["total_assets"]) * 100,
    np.nan
)

print("ROA calculated.")

ROA calculated.


In [12]:
print(
    data[
        [
            "ticker",
            "year",
            "revenue",
            "net_income",
            "total_assets",
            "profit_margin",
            "roa"
        ]
    ].head(20).to_string(index=False)
)

ticker  year      revenue   net_income  total_assets  profit_margin       roa
  AAPL  2015          NaN          NaN           NaN            NaN       NaN
  AAPL  2016          NaN          NaN           NaN            NaN       NaN
  AAPL  2017          NaN          NaN           NaN            NaN       NaN
  AAPL  2018          NaN          NaN           NaN            NaN       NaN
  AAPL  2019          NaN          NaN           NaN            NaN       NaN
  AAPL  2020          NaN          NaN           NaN            NaN       NaN
  AAPL  2021          NaN          NaN           NaN            NaN       NaN
  AAPL  2022 3.943280e+11 9.980300e+10  3.527550e+11      25.309641 28.292441
  AAPL  2023 3.832850e+11 9.699500e+10  3.525830e+11      25.306234 27.509835
  AAPL  2024 3.910350e+11 9.373600e+10  3.649800e+11      23.971256 25.682503
  AAPL  2025 4.161610e+11 1.120100e+11  3.592410e+11      26.915064 31.179626
   JNJ  2015          NaN          NaN           NaN            

In [13]:
print(
    data[
        [
            "revenue",
            "net_income",
            "total_assets",
            "profit_margin",
            "roa"
        ]
    ].isna().sum()
)

revenue          73
net_income       73
total_assets     73
profit_margin    73
roa              73
dtype: int64


In [14]:
data.to_csv(
    "../datasets/processed/day14_financial_ratios_dataset.csv",
    index=False
)

print("Day 14 dataset saved successfully.")

Day 14 dataset saved successfully.


In [15]:
import pandas as pd
import numpy as np

In [16]:
data = pd.read_csv(
    "../datasets/processed/day14_financial_ratios_dataset.csv"
)

print("Shape:", data.shape)
print(data.columns.tolist())

Shape: (110, 19)
['company', 'ticker', 'year', 'year_end_price', 'dividend_per_share', 'dividend_yield', 'dividend_growth', 'dividend_growth_category', 'annual_stock_return', 'return_category', 'total_return', 'return_volatility', 'average_return', 'dividend_consistency', 'revenue', 'net_income', 'total_assets', 'profit_margin', 'roa']


In [17]:
print(
    data[
        [
            "revenue",
            "net_income",
            "total_assets",
            "profit_margin",
            "roa"
        ]
    ].describe()
)

            revenue    net_income  total_assets  profit_margin        roa
count  3.700000e+01  3.700000e+01  3.700000e+01      37.000000  37.000000
mean   2.013961e+11  3.200059e+10  2.444649e+11      19.703639  12.542455
std    1.848379e+11  3.279848e+10  1.548482e+11      10.421290   6.900867
min    2.318200e+10  6.177000e+09  5.043560e+10       1.910717   3.054266
25%    8.200600e+10  1.063100e+10  1.005490e+11      10.427417   7.672325
50%    9.419300e+10  1.597400e+10  1.992100e+11      18.952589  10.965886
75%    3.346970e+11  3.515300e+10  3.763170e+11      26.915064  15.083620
max    6.809850e+11  1.120100e+11  6.190030e+11      41.279254  31.179626


In [19]:
from sklearn.preprocessing import MinMaxScaler

normalization_columns = [
    "dividend_yield",
    "dividend_growth",
    "annual_stock_return",
    "total_return",
    "return_volatility",
    "profit_margin",
    "roa"
]

available_columns = [
    col for col in normalization_columns
    if col in data.columns
]

scaler = MinMaxScaler()

for col in available_columns:
    data[col + "_normalized"] = scaler.fit_transform(
        data[[col]]
    )

print("Normalized columns:")
print([col for col in data.columns if "_normalized" in col])

Normalized columns:
['dividend_yield_normalized', 'dividend_growth_normalized', 'annual_stock_return_normalized', 'total_return_normalized', 'return_volatility_normalized', 'profit_margin_normalized', 'roa_normalized']


In [20]:
data[
    [col for col in data.columns if "_normalized" in col]
].head()

,dividend_yield_normalized,dividend_growth_normalized,annual_stock_return_normalized,total_return_normalized,return_volatility_normalized,profit_margin_normalized,roa_normalized
0,0.192308,NaN,NaN,NaN,1.0,NaN,NaN
1,0.192308,0.666667,0.400976,0.371376,1.0,NaN,NaN
2,0.132754,0.698745,0.684869,0.668811,1.0,NaN,NaN
3,0.174938,1.000000,0.268629,0.229677,1.0,NaN,NaN
4,0.081886,0.523710,1.000000,1.000000,1.0,NaN,NaN


In [21]:
data.to_csv(
    "../datasets/processed/day9_normalized_dataset.csv",
    index=False
)

print("Day 9 normalization completed.")

Day 9 normalization completed.


In [22]:
print([col for col in data.columns if "_normalized" in col])

['dividend_yield_normalized', 'dividend_growth_normalized', 'annual_stock_return_normalized', 'total_return_normalized', 'return_volatility_normalized', 'profit_margin_normalized', 'roa_normalized']


In [23]:
data.to_csv(
    "../datasets/processed/day9_normalized_dataset.csv",
    index=False
)

print("Day 9 normalization completed.")

Day 9 normalization completed.
